In [1]:
import pandas as pd
import os

path = r'C:\Users\ROOT\Desktop\churn-analysis\data\raw\\'

files = os.listdir(path)

print(f"Total files found: {len(files)}\n")
print(f"{'File Name':<55} {'Rows':>7} {'Cols':>5} {'Nulls':>7}")
print("-" * 78)

for f in files:
    if f.endswith('.csv'):
        df = pd.read_csv(path + f)
        nulls = df.isnull().sum().sum()
        print(f"{f:<55} {df.shape[0]:>7,} {df.shape[1]:>5} {nulls:>7,}")

Total files found: 9

File Name                                                  Rows  Cols   Nulls
------------------------------------------------------------------------------
olist_customers_dataset.csv                              99,441     5       0
olist_geolocation_dataset.csv                           1,000,163     5       0
olist_orders_dataset.csv                                 99,441     8   4,908
olist_order_items_dataset.csv                           112,650     7       0
olist_order_payments_dataset.csv                        103,886     5       0
olist_order_reviews_dataset.csv                          99,224     7 145,903
olist_products_dataset.csv                               32,951     9   2,448
olist_sellers_dataset.csv                                 3,095     4       0
product_category_name_translation.csv                        71     2       0


In [3]:
# Loading the 3 files with nulls — understanding them before cleaning
import pandas as pd

path = r'C:\Users\ROOT\Desktop\churn-analysis\data\raw\\'

orders = pd.read_csv(path + 'olist_orders_dataset.csv')
reviews = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(path + 'olist_products_dataset.csv')

# --- ORDERS: which columns have nulls? ---
print("=== ORDERS — Null breakdown ===")
print(orders.isnull().sum())
print("\nOrder status breakdown:")
print(orders['order_status'].value_counts())

# --- REVIEWS: which columns have nulls? ---
print("\n=== REVIEWS — Null breakdown ===")
print(reviews.isnull().sum())

# --- PRODUCTS: which columns have nulls? ---
print("\n=== PRODUCTS — Null breakdown ===")
print(products.isnull().sum())

=== ORDERS — Null breakdown ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status breakdown:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

=== REVIEWS — Null breakdown ===
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

=== PRODUCTS — Null breakdown ===
product_id                      0
product_category_name         610
product_name_lenght           610
prod

In [1]:
import pandas as pd
import os

path = r'C:\Users\ROOT\Desktop\churn-analysis\data\raw\\'
out  = r'C:\Users\ROOT\Desktop\churn-analysis\data\cleaned\\'

# ── Loading all 9 files ──────────────────────────────────────────
orders      = pd.read_csv(path + 'olist_orders_dataset.csv',
                parse_dates=['order_purchase_timestamp',
                             'order_approved_at',
                             'order_delivered_carrier_date',
                             'order_delivered_customer_date',
                             'order_estimated_delivery_date'])

customers   = pd.read_csv(path + 'olist_customers_dataset.csv')
payments    = pd.read_csv(path + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
items       = pd.read_csv(path + 'olist_order_items_dataset.csv',
                parse_dates=['shipping_limit_date'])
products    = pd.read_csv(path + 'olist_products_dataset.csv')
sellers     = pd.read_csv(path + 'olist_sellers_dataset.csv')
translation = pd.read_csv(path + 'product_category_name_translation.csv')

print("✅ All 9 files loaded")

# ── Cleaning ORDERS ──────────────────────────────────────────────
# Filling 160 missing approval times with purchase time
orders['order_approved_at'] = orders['order_approved_at'].fillna(
    orders['order_purchase_timestamp'])

# Flaging undelivered orders (don't delete — it matter for churn)
orders['is_delivered'] = orders['order_delivered_customer_date'].notna().astype(int)

# Delivery delay in days (negative = early, positive = late)
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.days

print("✅ Orders cleaned — delivery delay column created")

# ── Clean PRODUCTS ────────────────────────────────────────────
# Joining English category names
products = products.merge(translation, on='product_category_name', how='left')

# Filling missing category with 'unknown'
products['product_category_name_english'] = (
    products['product_category_name_english'].fillna('unknown'))

# Filling 2 missing dimensions with median
for col in ['product_weight_g','product_length_cm',
            'product_height_cm','product_width_cm']:
    products[col] = products[col].fillna(products[col].median())

print("✅ Products cleaned — English categories added")

# ── Aggregating payments & items per order ─────────────────────
pay_agg = (payments.groupby('order_id')
           .agg(total_payment=('payment_value','sum'),
                payment_installments=('payment_installments','max'),
                payment_type=('payment_type','first'))
           .reset_index())

items_agg = (items.groupby('order_id')
             .agg(total_items=('order_item_id','count'),
                  total_price=('price','sum'),
                  total_freight=('freight_value','sum'))
             .reset_index())

print("✅ Payments & items aggregated per order")

# ── Building MASTER TABLE ────────────────────────────────────────
master = (orders
    .merge(customers,  on='customer_id',  how='left')
    .merge(pay_agg,    on='order_id',     how='left')
    .merge(items_agg,  on='order_id',     how='left')
    .merge(reviews[['order_id','review_score']], on='order_id', how='left')
)

print("✅ Master table built")
print(f"   Shape: {master.shape}")

# ── Building RFM TABLE ───────────────────────────────────────────
snapshot_date = master['order_purchase_timestamp'].max()

rfm = (master
    .groupby('customer_unique_id')
    .agg(
        Recency   = ('order_purchase_timestamp',
                     lambda x: (snapshot_date - x.max()).days),
        Frequency = ('order_id', 'count'),
        Monetary  = ('total_payment', 'sum')
    )
    .reset_index()
)

print(f"✅ RFM table built — {rfm.shape[0]:,} unique customers")

# ── RFM SCORING (1–5) ─────────────────────────────────────────
rfm['R_Score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),
                          5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  5, labels=[1,2,3,4,5]).astype(int)

def segment(row):
    r, f = row['R_Score'], row['F_Score']
    if   r >= 4 and f >= 4: return 'Champion'
    elif r >= 3 and f >= 3: return 'Loyal'
    elif r >= 3 and f <= 2: return 'Potential Loyalist'
    elif r <= 2 and f >= 3: return 'At Risk'
    else:                   return 'Lost'

rfm['Segment'] = rfm.apply(segment, axis=1)

print("\n=== RFM Segment Distribution ===")
print(rfm['Segment'].value_counts())
print(f"\nTotal customers: {rfm.shape[0]:,}")

# ── Saving both tables ──────────────────────────────────────────
os.makedirs(out, exist_ok=True)
master.to_csv(out + 'master_table.csv', index=False)
rfm.to_csv(out + 'rfm_table.csv', index=False)

print(f"\n✅ master_table.csv saved → {master.shape}")
print(f"✅ rfm_table.csv saved    → {rfm.shape}")
print("\n🎉 Phase 2 complete — ready for EDA!")

✅ All 9 files loaded
✅ Orders cleaned — delivery delay column created
✅ Products cleaned — English categories added
✅ Payments & items aggregated per order
✅ Master table built
   Shape: (99992, 21)
✅ RFM table built — 96,096 unique customers

=== RFM Segment Distribution ===
Segment
Potential Loyalist    22975
At Risk               22966
Loyal                 19250
Lost                  15464
Champion              15441
Name: count, dtype: int64

Total customers: 96,096

✅ master_table.csv saved → (99992, 21)
✅ rfm_table.csv saved    → (96096, 8)

🎉 Phase 2 complete — ready for EDA!
